In [ ]:
from atlas.common.spark.bootstrap_initialization import initialize_atlas

settings, spark = initialize_atlas()

In [ ]:

from pyspark.sql.types import DateType, LongType, StringType, StructField, StructType, TimestampType

customer_snapshot_schema = StructType([
        StructField("customer_id", LongType(), False),
        StructField("first_name", StringType(), False),
        StructField("last_name", StringType(), False),
        StructField("email", StringType(), True),
        StructField("phone_number", StringType(), True),
        StructField("date_of_birth", DateType(), True),
        StructField("status", StringType(), False),
        StructField("segment", StringType(), False),
        StructField("created_at", TimestampType(), False),
        StructField("updated_at", TimestampType(), False),
    ])

In [ ]:
snapshot_path = "/misc/customer_snapshot_2026-09-13.csv"

In [ ]:
snapshot_df = spark.read.option("header", True).schema(customer_snapshot_schema).csv(snapshot_path)

In [ ]:
snapshot_df.printSchema()
snapshot_df.show(truncate=False)

In [ ]:
from pyspark.sql import functions as F

snapshot_as_of = "2026-09-12 00:00:00"
snapshot_metadata = (
    snapshot_df
    .withColumn(
        "snapshot_as_of",
        F.to_timestamp(F.lit(snapshot_as_of))
    )
    .withColumn(
        "source_date",
        F.to_date(F.col("snapshot_as_of"))
    )
    .withColumn(
        "batch_id",
        F.concat_ws(
            "_",
            F.lit("customer_snapshot"),
            F.date_format(F.col("snapshot_as_of"), "yyyyMMdd"),
        )
    )
    .withColumn(
        "source_filename",
        F.concat(
            F.lit("customers_"),
            F.date_format(F.col("snapshot_as_of"), "yyyy-MM-dd"),
            F.lit(".csv"),
        )
    )
    .withColumn("schema_version", F.lit(1))
    .withColumn("ingested_at", F.current_timestamp())
)

In [ ]:
customer_snapshot_conditions = (F.array(
        F.when(F.col("customer_id").isNull(), F.lit("MISSING_CUSTOMER_ID")),
        F.when((F.col("first_name").isNull() | (F.trim(F.col("first_name")) == "")), F.lit("MISSING_FIRST_NAME")),
        F.when((F.col("last_name").isNull() | (F.trim(F.col("last_name")) == "")), F.lit("MISSING_LAST_NAME")),
        F.when((F.col("email").isNull()| (F.trim(F.col("email")) == ""))&
            (F.col("phone_number").isNull()| (F.trim(F.col("phone_number")) == "")),
            F.lit("MISSING_CONTACT_INFO")),
        F.when(F.col("date_of_birth") > F.current_date(), F.lit("FUTURE_DATE_OF_BIRTH")),
        F.when(F.col("status").isNull() | ~F.col("status").isin(["ACTIVE", "INACTIVE", "SUSPENDED"]), F.lit("INVALID_STATUS")),
        F.when(F.col("segment").isNull() |~F.col("segment").isin(["STANDARD", "GOLD", "PREMIUM"]), F.lit("INVALID_SEGMENT"))
    ))

In [ ]:
snapshot_filtered = snapshot_metadata.withColumn("snapshot_errors", F.array_compact(customer_snapshot_conditions))

In [ ]:
snapshot_valid_records = snapshot_filtered.filter(F.size(F.col("snapshot_errors")) ==0).drop("snapshot_errors")

In [ ]:
snapshot_duplicate_rows = snapshot_valid_records.groupBy("customer_id").agg(F.count('*').alias("total_count")).filter(F.col("total_count") >1)
snapshot_duplicate_rows.show()

In [ ]:
snapshot_duplicate_records = snapshot_valid_records.join(snapshot_duplicate_rows, "customer_id", "left_semi")
snapshot_reconciliation_ready = snapshot_valid_records.join(snapshot_duplicate_rows, "customer_id", "left_anti")

In [ ]:
snapshot_duplicate_records.show()

In [ ]:
snapshot_reconciliation_ready.show()

In [ ]:
from atlas.common.paths.get_cdc_paths import get_silver_paths

silver_customer_history_path = get_silver_paths(settings, "customer", "customers", "cdc_history")

In [ ]:
from delta import DeltaTable

customer_cdc_history = DeltaTable.forPath(spark,silver_customer_history_path).toDF()

In [ ]:
customer_cdc_history.printSchema()

In [ ]:
cdc_as_of_snapshot = customer_cdc_history.filter(F.col("source_timestamp")<= F.to_timestamp(F.lit(snapshot_as_of)))

In [ ]:
from pyspark.sql.window import Window

customer_as_of_window = Window.partitionBy("customer_id").orderBy(F.col("source_lsn").desc())
expected_customer_state = cdc_as_of_snapshot.withColumn("rn", F.row_number().over(customer_as_of_window)).filter(F.col("rn") ==1).drop("rn")
expected_customer_state = expected_customer_state.filter(F.col("cdc_operation") != "d")

In [ ]:
expected_customer_state.show()

In [ ]:
bucket_count = 4

In [ ]:
snapshot_bucketed = snapshot_reconciliation_ready.withColumn("bucket_id", F.pmod(F.hash(F.col("customer_id")), F.lit(bucket_count))+1)
expected_bucketed = expected_customer_state.withColumn("bucket_id", F.pmod(F.hash(F.col("customer_id")), F.lit(bucket_count))+1)

In [ ]:
snapshot_bucketed.select("customer_id", "bucket_id").show()
expected_bucketed.select("customer_id", "bucket_id").show()

In [ ]:
expected_bucketed.printSchema()

In [ ]:
normalized_reconciliation_columns = [
    F.coalesce(F.col("customer_id").cast("string"), F.lit("__NULL__")),
    F.coalesce(F.trim(F.col("first_name")),F.lit("__NULL__")),
    F.coalesce(F.trim(F.col("last_name")),F.lit("__NULL__")),
    F.coalesce(F.lower(F.trim(F.col("email"))),F.lit("__NULL__")),
    F.coalesce(F.trim(F.col("phone_number")),F.lit("__NULL__")),
    F.coalesce(F.date_format(F.col("date_of_birth"), "yyyy-MM-dd"),F.lit("__NULL__")),
    F.coalesce(F.upper(F.trim(F.col("status"))),F.lit("__NULL__")),
    F.coalesce(F.upper(F.trim(F.col("segment"))),F.lit("__NULL__")),
    F.coalesce(F.date_format(F.col("updated_at"),"yyyy-MM-dd HH:mm:ss.SSSSSS"),F.lit("__NULL__"))]

In [ ]:
snapshot_with_checksum = snapshot_bucketed.withColumn("row_checksum", F.sha2(F.concat_ws("||", *normalized_reconciliation_columns), 256))
expected_with_checksum = expected_bucketed.withColumn("row_checksum", F.sha2(F.concat_ws("||", *normalized_reconciliation_columns), 256))

In [ ]:
snapshot_with_checksum.select(
    "customer_id",
    "bucket_id",
    "row_checksum"
).show(truncate=False)

expected_with_checksum.select(
    "customer_id",
    "bucket_id",
    "row_checksum"
).show(truncate=False)

In [ ]:
snapshot_bucket_checksum = (snapshot_with_checksum.groupBy(F.col("bucket_id"))
                            .agg(F.sha2(F.concat_ws("||", F.sort_array(F.collect_list("row_checksum"))), 256).alias("snapshot_checksum"),
                                 F.count(F.col("customer_id")).alias("snapshot_row_count")))

expected_bucket_checksum = (expected_with_checksum.groupBy(F.col("bucket_id"))
                            .agg(F.sha2(F.concat_ws("||", F.sort_array(F.collect_list("row_checksum"))), 256).alias("expected_checksum"),
                                 F.count(F.col("customer_id")).alias("expected_row_count")))

In [ ]:
snapshot_bucket_checksum.show()
expected_bucket_checksum .show()

In [ ]:
snap_exp_join = snapshot_bucket_checksum.join(expected_bucket_checksum, on="bucket_id", how="full_outer")

In [ ]:
snap_exp_join.show()

In [ ]:
reconciliation_status = F.when(
        F.col("snapshot_row_count").isNull() & (~F.col("expected_row_count").isNull()), F.lit("MISSING_IN_SNAPSHOT")
    ).when(
        F.col("expected_row_count").isNull() & (~F.col("snapshot_row_count").isNull()), F.lit("MISSING_IN_CDC")
    ).when(
        F.col("snapshot_row_count") != F.col("expected_row_count"), F.lit("COUNT_MISMATCH")
    ).when(
        F.col("snapshot_checksum") != F.col("expected_checksum"), F.lit("CHECKSUM_MISMATCH")
    ).otherwise(F.lit("MATCH"))

In [ ]:
snap_exp_rec_status = snap_exp_join.withColumn("reconciliation_status", reconciliation_status)

In [ ]:
matched_buckets = snap_exp_rec_status.filter(F.col("reconciliation_status")=="MATCH")
unmatched_buckets = snap_exp_rec_status.filter(F.col("reconciliation_status")!="MATCH")

In [ ]:
matched_buckets.show()
unmatched_buckets.show()

In [ ]:
snapshot_unmatched_rows = (
    snapshot_with_checksum
    .join(
        unmatched_buckets.select("bucket_id"),
        on="bucket_id",
        how="left_semi"
    )
    .withColumnRenamed("row_checksum", "snapshot_row_checksum")
    .withColumn("snapshot_present", F.lit(True))
)

expected_unmatched_rows = (
    expected_with_checksum
    .join(
        unmatched_buckets.select("bucket_id"),
        on="bucket_id",
        how="left_semi"
    )
    .withColumnRenamed("row_checksum", "expected_row_checksum")
    .withColumn("expected_present", F.lit(True))
)


In [ ]:
snapshot_unmatched_rows.show(vertical=True,)


In [ ]:
expected_unmatched_rows.show(vertical=True,)

In [ ]:
reconciliation_row_status = (
    F.when(
        F.col("snapshot_present").isNull()
        & F.col("expected_present").isNotNull(),
        F.lit("MISSING_IN_SNAPSHOT")
    )
    .when(
        F.col("expected_present").isNull()
        & F.col("snapshot_present").isNotNull(),
        F.lit("MISSING_IN_CDC")
    )
    .when(
        F.col("snapshot_row_checksum") != F.col("expected_row_checksum"),
        F.lit("CHECKSUM_MISMATCH")
    )
    .otherwise(F.lit("MATCH"))
)

In [ ]:
joined_snap_exp_df = (
    snapshot_unmatched_rows.alias("s")
    .join(
        expected_unmatched_rows.alias("e"),
        on="customer_id",
        how="full_outer"
    )
    .withColumn(
        "reconciliation_status",
        reconciliation_row_status
    )
)

In [ ]:
joined_snap_exp_df.show(vertical=True,)

In [ ]:
reconciled_unmatched_rows = joined_snap_exp_df.filter(F.col("reconciliation_status")!="MATCH")
reconciled_matched_rows = joined_snap_exp_df.filter(F.col("reconciliation_status")=="MATCH")

In [ ]:
customer_reconciliation_columns = ["first_name","last_name", "email","phone_number", "date_of_birth", "status", "segment", "updated_at"]

In [ ]:
snapshot_columns = [
    F.col(f"s.{col}").alias(f"snapshot_{col}")
    for col in customer_reconciliation_columns
]

expected_columns = [
    F.col(f"e.{col}").alias(f"expected_{col}")
    for col in customer_reconciliation_columns
]

In [ ]:
reconciliation_detail = reconciled_unmatched_rows.select(
    F.col("customer_id"),
    F.col("s.snapshot_row_checksum"),
    F.col("e.expected_row_checksum"),
    F.col("snapshot_present"),
    F.col("expected_present"),
    F.col("reconciliation_status"),
    *snapshot_columns,
    *expected_columns
)

In [ ]:
reconciliation_detail.show()

In [ ]:
mismatch_checksum_rows = reconciliation_detail.filter(F.col("reconciliation_status")=="CHECKSUM_MISMATCH")

In [ ]:
missing_rows = reconciliation_detail.filter(F.col("reconciliation_status")!="CHECKSUM_MISMATCH")

In [ ]:
def mismatch_reasons(customer_reconciliation_columns):
    mismatch_reason = []

    for col in customer_reconciliation_columns:
        mismatch_reason.append(
            F.when(
                ~F.col(f"snapshot_{col}").eqNullSafe(
                    F.col(f"expected_{col}")
                ),
                F.lit(f"{col.upper()}_MISMATCH")
            )
        )

    return F.array_compact(
        F.array(*mismatch_reason)
    )


In [ ]:
reconciled_customer_mismatch = mismatch_checksum_rows.withColumn("mismatch_columns",  mismatch_reasons(customer_reconciliation_columns))

In [ ]:
reconciled_customer_mismatch.show(vertical=True,truncate =False)

In [ ]:
missing_rows.printSchema()
reconciled_customer_mismatch.printSchema()

In [ ]:
missing_rows = missing_rows.withColumn(
    "mismatch_columns",
    F.array().cast("array<string>")
)

In [ ]:
final_reconciliation_exceptions = missing_rows.unionByName(
    reconciled_customer_mismatch
)

In [ ]:
final_reconciliation_exceptions.printSchema()

In [59]:
import uuid

reconciliation_run_id = str(uuid.uuid4())

final_reconciliation_exceptions = (
    final_reconciliation_exceptions
    .withColumn(
        "reconciliation_run_id",
        F.lit(reconciliation_run_id)
    )
    .withColumn(
        "snapshot_as_of",
        F.to_timestamp(F.lit(snapshot_as_of))
    )
    .withColumn(
        "reconciled_at",
        F.current_timestamp()
    )
    .withColumn(
        "entity_name",
        F.lit("customer")
    )
)

In [60]:
snapshot_run_metrics = snapshot_reconciliation_ready.agg(
    F.count("*").alias("snapshot_row_count")
)

cdc_run_metrics = expected_customer_state.agg(
    F.count("*").alias("cdc_row_count")
)

exception_run_metrics = final_reconciliation_exceptions.agg(
    F.count("*").alias("exception_count"),

    F.sum(
        F.when(
            F.col("reconciliation_status") == "MISSING_IN_CDC", 1
        ).otherwise(0)
    ).alias("missing_in_cdc_count"),

    F.sum(
        F.when(
            F.col("reconciliation_status") == "MISSING_IN_SNAPSHOT", 1
        ).otherwise(0)
    ).alias("missing_in_snapshot_count"),

    F.sum(
        F.when(
            F.col("reconciliation_status") == "CHECKSUM_MISMATCH", 1
        ).otherwise(0)
    ).alias("checksum_mismatch_count")
)

In [61]:
reconciliation_run_summary = (
    snapshot_run_metrics
    .crossJoin(cdc_run_metrics)
    .crossJoin(exception_run_metrics)
    .withColumn(
        "matched_row_count",
        F.col("snapshot_row_count")
        - F.col("missing_in_cdc_count")
        - F.col("checksum_mismatch_count")
    )
    .withColumn(
        "overall_status",
        F.when(
            F.col("exception_count") == 0,
            F.lit("SUCCESS")
        ).otherwise(
            F.lit("COMPLETED_WITH_EXCEPTIONS")
        )
    )
    .withColumn(
        "reconciliation_run_id",
        F.lit(reconciliation_run_id)
    )
    .withColumn(
        "entity_name",
        F.lit("customer")
    )
    .withColumn(
        "snapshot_as_of",
        F.to_timestamp(F.lit(snapshot_as_of))
    )
    .withColumn(
        "reconciled_at",
        F.current_timestamp()
    )
)

In [63]:
reconciliation_run_summary.show(vertical =True)

-RECORD 0-----------------------------------------
 snapshot_row_count        | 3                    
 cdc_row_count             | 7                    
 exception_count           | 9                    
 missing_in_cdc_count      | 2                    
 missing_in_snapshot_count | 6                    
 checksum_mismatch_count   | 1                    
 matched_row_count         | 0                    
 overall_status            | COMPLETED_WITH_EX... 
 reconciliation_run_id     | 9c508c46-3c03-413... 
 entity_name               | customer             
 snapshot_as_of            | 2026-09-12 00:00:00  
 reconciled_at             | 2026-09-13 02:15:... 

